<a href="https://colab.research.google.com/github/Alok224/Celebal_Weekly_Assignments/blob/main/Week8_Alok_Assignment8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-Agent Smart Assistant using Intent Routing and Tool Calling

**Objective:** Build a single-agent AI assistant that understands a user query, figures out the intent behind it, routes the query to the right tool, and returns the result as structured JSON.



In [1]:
import re
import json
import math
import string
from datetime import datetime

In [2]:
def calculator(expression):
    """Safely evaluates a basic math expression."""
    allowed_chars = set("0123456789+-*/(). ")

    if not expression or not expression.strip():
        raise ValueError("No expression found to calculate")

    if not all(ch in allowed_chars for ch in expression):
        raise ValueError("Expression contains invalid characters")

    try:
        result = eval(expression, {"__builtins__": {}}, {})
    except ZeroDivisionError:
        raise ZeroDivisionError("Division by zero is not allowed")
    except Exception:
        raise ValueError("Invalid mathematical expression")

    return result

In [3]:
STOP_WORDS = {
    "is", "am", "are", "was", "were", "the", "a", "an", "and", "or", "but",
    "of", "in", "on", "at", "to", "for", "from", "with", "about", "into",
    "this", "that", "these", "those", "it", "its", "as", "by", "be", "has",
    "have", "had", "will", "shall", "can", "could", "would", "should"
}

def extract_keywords(text):
    """Extracts unique, meaningful keywords from a sentence."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()

    keywords = [w for w in words if w not in STOP_WORDS]
    unique_keywords = list(dict.fromkeys(keywords))

    return unique_keywords

In [4]:
def general_response(query):
    """Returns a canned response for casual/conversational queries."""
    query_lower = query.lower().strip()

    greetings = ["hello", "hi", "hey", "good morning", "good evening"]
    if any(greet in query_lower for greet in greetings):
        return "Hello! How can I assist you today?"

    if "how are you" in query_lower:
        return "I am just a program, but I am running fine!"

    if "your name" in query_lower:
        return "I am a Single-Agent Smart Assistant."

    if "thank" in query_lower:
        return "You are welcome!"

    return "I am not sure how to respond to that, but I am here to help."


def word_counter(text):
    """Bonus tool: counts number of words in the given text."""
    words = text.split()
    return len(words)

In [5]:
def extract_math_expression(query):
    """Pulls out only the math part from a query like 'calculate 25*8'."""
    query_lower = query.lower().replace("calculate", "")
    expression = re.sub(r"[^0-9+\-*/(). ]", "", query_lower)
    return expression.strip()


def extract_text_for_keywords(query):
    """Pulls out the actual sentence to run keyword extraction on."""
    match = re.search(r"keywords?\s*(from|in|of)?\s*(.*)", query.lower())
    if match and match.group(2):
        return match.group(2).strip()
    return query


def log_request(query):
    """Bonus feature: logs every request with a timestamp."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] Query received: {query}")


def agent(query):
    """Main agent function. Routes the query to the correct tool based on intent."""
    log_request(query)

    if not query or not query.strip():
        return {"type": "error", "result": "Input cannot be empty"}

    query_lower = query.lower().strip()

    try:
        if "calculate" in query_lower:
            expression = extract_math_expression(query)
            result = calculator(expression)
            return {"type": "calculation", "result": result}

        elif "keyword" in query_lower:
            text = extract_text_for_keywords(query)
            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        elif "count words" in query_lower or "word count" in query_lower:
            text = re.sub(r"(count words|word count)", "", query_lower).strip()
            count = word_counter(text)
            return {"type": "word_count", "result": count}

        else:
            response = general_response(query)
            return {"type": "general", "result": response}

    except ZeroDivisionError as e:
        return {"type": "error", "result": str(e)}
    except ValueError as e:
        return {"type": "error", "result": str(e)}
    except Exception as e:
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

In [6]:
sample_output = agent("calculate 12+8")
print(json.dumps(sample_output, indent=4))

[2026-07-12 10:33:57] Query received: calculate 12+8
{
    "type": "calculation",
    "result": 20
}


In [7]:
sample_queries = [
    "calculate 25*8",
    "extract keywords from Artificial Intelligence is changing education",
    "Hello",
    "calculate 10/0",
    "keywords Python is a programming language",
    ""
]

for q in sample_queries:
    output = agent(q)
    print(json.dumps(output, indent=4))
    print("-" * 40)

[2026-07-12 10:33:57] Query received: calculate 25*8
{
    "type": "calculation",
    "result": 200
}
----------------------------------------
[2026-07-12 10:33:57] Query received: extract keywords from Artificial Intelligence is changing education
{
    "type": "keywords",
    "result": [
        "artificial",
        "intelligence",
        "changing",
        "education"
    ]
}
----------------------------------------
[2026-07-12 10:33:57] Query received: Hello
{
    "type": "general",
    "result": "Hello! How can I assist you today?"
}
----------------------------------------
[2026-07-12 10:33:57] Query received: calculate 10/0
{
    "type": "error",
    "result": "Division by zero is not allowed"
}
----------------------------------------
[2026-07-12 10:33:57] Query received: keywords Python is a programming language
{
    "type": "keywords",
    "result": [
        "python",
        "programming",
        "language"
    ]
}
----------------------------------------
[2026-07-12 1

In [9]:
while True:
    user_query = input("Enter your query (type 'exit' to quit): ")

    if user_query.strip().lower() == "exit":
        print("Exiting Smart Assistant. Goodbye!")
        break

    output = agent(user_query)
    print(json.dumps(output, indent=4))

Enter your query (type 'exit' to quit): Hello, How are you?
[2026-07-12 10:37:34] Query received: Hello, How are you?
{
    "type": "general",
    "result": "Hello! How can I assist you today?"
}
Enter your query (type 'exit' to quit): Calculate 12/4
[2026-07-12 10:37:41] Query received: Calculate 12/4
{
    "type": "calculation",
    "result": 3.0
}
Enter your query (type 'exit' to quit): Calculate 10/0
[2026-07-12 10:37:50] Query received: Calculate 10/0
{
    "type": "error",
    "result": "Division by zero is not allowed"
}
Enter your query (type 'exit' to quit): exit
Exiting Smart Assistant. Goodbye!


## Conclusion

This notebook implements a simple single-agent pipeline:

- **Understanding the query** – the agent looks at the raw text typed by the user.
- **Intent routing** – based on keywords like "calculate" or "keyword", the query is sent to the right tool using plain if/elif checks (no ML, no LLM).
- **Tool integration** – three tools are used: a calculator, a keyword extractor, and a bonus word counter, along with a general response handler for casual chat.
- **Structured output** – every response, regardless of which tool handled it, follows the same `{"type": ..., "result": ...}` JSON schema, making it easy for any frontend to consume.
- **Error handling** – empty inputs, divide-by-zero, and invalid expressions are all caught and returned as a clean `"error"` type instead of crashing the program.

**Future improvements:**
- Use NLP libraries (like spaCy) for smarter intent detection instead of keyword matching.
- Add more tools, such as unit converters or date/time calculators.
- Store conversation history to enable follow-up/contextual queries.
- Replace rule-based routing with a lightweight ML classifier for intent detection.